K-Nearest Neighbors Classification Predict either **gammas (signal)** or **hadrons (background)** using the MAGIC gamma telescope dataset.

In [25]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

#test our model
from sklearn.metrics import confusion_matrix , accuracy_score ,precision_score ,recall_score, f1_score

In [5]:
dataSet = pd.read_csv('magic04.data' , header=None)
dataSet

#12332 gamma events and 6688 hadron events.  



,0,1,2,3,4,5,6,7,8,9,10
0,28.7967,16.0021,2.6449,0.3918,0.1982,27.7004,22.0110,-8.2027,40.0920,81.8828,g
1,31.6036,11.7235,2.5185,0.5303,0.3773,26.2722,23.8238,-9.9574,6.3609,205.2610,g
2,162.0520,136.0310,4.0612,0.0374,0.0187,116.7410,-64.8580,-45.2160,76.9600,256.7880,g
3,23.8172,9.5728,2.3385,0.6147,0.3922,27.2107,-6.4633,-7.1513,10.4490,116.7370,g
4,75.1362,30.9205,3.1611,0.3168,0.1832,-5.5277,28.5525,21.8393,4.6480,356.4620,g
...,...,...,...,...,...,...,...,...,...,...,...
19015,21.3846,10.9170,2.6161,0.5857,0.3934,15.2618,11.5245,2.8766,2.4229,106.8258,h
19016,28.9452,6.7020,2.2672,0.5351,0.2784,37.0816,13.1853,-2.9632,86.7975,247.4560,h
19017,75.4455,47.5305,3.4483,0.1417,0.0549,-9.3561,41.0562,-9.4662,30.2987,256.5166,h
19018,120.5135,76.9018,3.9939,0.0944,0.0683,5.8043,-93.5224,-63.8389,84.6874,408.3166,h


In [7]:
#split data into features and labels
X = dataSet.iloc[: , :-1]
Y = dataSet.iloc[: , -1]

Y.value_counts()


10
g    12332
h     6688
Name: count, dtype: int64

In [15]:
#Random Under Sampling
from imblearn.under_sampling import RandomUnderSampler
rus = RandomUnderSampler(sampling_strategy=1)
X_resampled_RUS , Y_resampled_RUS = rus.fit_resample(X , Y)
X_resampled_RUS.value_counts()

0         1         2       3       4       5          6          7         8        9       
38.5399   20.4511   2.7214  0.3324  0.2194  -2.0651     15.7963   -19.8976  86.2652  260.9730    2
20.8382   10.2682   2.3404  0.6210  0.3356   4.0453    -15.9238   -2.9230   86.8812  222.9860    2
19.4865   7.3947    2.5058  0.7551  0.4321  -2.3691     8.6595     4.5933   9.7680   181.3700    2
24.0108   17.2422   2.4669  0.4369  0.2440   14.2428    19.4721    17.2936  67.1405  142.8360    2
24.4126   11.7612   2.6274  0.4528  0.2700   9.1242     13.2079   -6.2050   53.7096  180.6770    2
                                                                                                ..
262.2030  82.2943   3.7197  0.1214  0.0619  -299.7380  -152.0160   34.0519  63.2988  223.4850    1
263.7030  117.1310  3.4602  0.1137  0.0608  -287.7720  -117.3880   90.0203  68.2010  165.3040    1
263.9140  256.3820  3.7684  0.0651  0.0326   220.4880   52.4448   -95.0139  10.3120  31.9174     1
265.5424  69.42

In [16]:
#Random Over Sampling
from imblearn.over_sampling import RandomOverSampler
ros = RandomOverSampler(sampling_strategy=1)
X_resampled_ROS , Y_resampled_ROS = ros.fit_resample(X , Y)
Y_resampled_ROS.value_counts()


10
g    12332
h    12332
Name: count, dtype: int64

In [17]:
#using Pandas to balance the dataset

gamma_data = dataSet[dataSet[10] == 'g']
hadron_data = dataSet[dataSet[10] == 'h']

gamma_data_underSampled = gamma_data.sample(n=len(hadron_data) , random_state=0)
balanced_data = pd.concat([gamma_data_underSampled , hadron_data])
balanced_data[10].value_counts()
balanced_data

,0,1,2,3,4,5,6,7,8,9,10
6697,69.2979,26.8809,3.1930,0.2065,0.1074,39.6296,44.3457,-23.0604,9.3234,248.7500,g
4583,24.5939,10.1418,2.5676,0.5007,0.2693,-8.4503,15.2452,-7.0283,17.0056,173.2880,g
7549,55.4800,27.1606,3.1826,0.2299,0.1225,43.1016,54.2556,13.7406,32.2220,262.1810,g
9117,12.6594,11.7413,2.1351,0.7033,0.3846,-15.8596,9.4522,-8.7126,43.5434,227.7110,g
2768,38.6204,20.5632,2.9770,0.2478,0.1270,-13.8229,-31.3983,-13.1337,5.8671,192.4670,g
...,...,...,...,...,...,...,...,...,...,...,...
19015,21.3846,10.9170,2.6161,0.5857,0.3934,15.2618,11.5245,2.8766,2.4229,106.8258,h
19016,28.9452,6.7020,2.2672,0.5351,0.2784,37.0816,13.1853,-2.9632,86.7975,247.4560,h
19017,75.4455,47.5305,3.4483,0.1417,0.0549,-9.3561,41.0562,-9.4662,30.2987,256.5166,h
19018,120.5135,76.9018,3.9939,0.0944,0.0683,5.8043,-93.5224,-63.8389,84.6874,408.3166,h


In [18]:
#split data into training and testing sets
X_train , X_temp , Y_train , Y_temp = train_test_split(X_resampled_RUS , Y_resampled_RUS , test_size=0.3 , random_state=0)
X_test , X_val , Y_test , Y_val = train_test_split(X_temp , Y_temp , test_size=0.5 , random_state=0)

In [19]:
#Feature Scaling
sc_X = StandardScaler()
X_train = sc_X.fit_transform(X_train)
X_test = sc_X.transform(X_test)
X_val = sc_X.transform(X_val)

In [29]:
K_values = [3 , 5 , 7 , 9 , 11]
for K in K_values:
    knn_classifier = KNeighborsClassifier(n_neighbors=K , p = 2 , metric='euclidean')
    knn_classifier.fit(X_train , Y_train)
    
    #first predict on the validation set:
    Y_val_pred = knn_classifier.predict(X_val)

    #Report all of your trained model accuracy, precision, recall and f-score as well as confusion matrix
    print(f'K = {K} : ')
    print("accuracy_score : " , accuracy_score(Y_val , Y_val_pred))
    print("percision_score : " , precision_score(Y_val , Y_val_pred , pos_label='g'))
    print("recall_score : " , recall_score(Y_val , Y_val_pred , pos_label='g'))
    print("f1_score : " , f1_score(Y_val , Y_val_pred , pos_label='g'))
    print("confusion_matrix : \n" , confusion_matrix(Y_val , Y_val_pred))

    
    

K = 3 : 
accuracy_score :  0.8111609367214748
percision_score :  0.7767857142857143
recall_score :  0.8708708708708709
f1_score :  0.8211420481359132
confusion_matrix : 
 [[870 129]
 [250 758]]
K = 5 : 
accuracy_score :  0.8161434977578476
percision_score :  0.7777777777777778
recall_score :  0.8828828828828829
f1_score :  0.8270042194092827
confusion_matrix : 
 [[882 117]
 [252 756]]
K = 7 : 
accuracy_score :  0.815146985550573
percision_score :  0.7730434782608696
recall_score :  0.8898898898898899
f1_score :  0.8273615635179153
confusion_matrix : 
 [[889 110]
 [261 747]]
K = 9 : 
accuracy_score :  0.8106626806178375
percision_score :  0.7675021607605877
recall_score :  0.8888888888888888
f1_score :  0.8237476808905381
confusion_matrix : 
 [[888 111]
 [269 739]]
K = 11 : 
accuracy_score :  0.8106626806178375
percision_score :  0.766122098022356
recall_score :  0.8918918918918919
f1_score :  0.8242368177613321
confusion_matrix : 
 [[891 108]
 [272 736]]


In [ ]:
# predict the test set results
classifier = KNeighborsClassifier(n_neighbors=7 , p = 2 , metric='euclidean')
classifier.fit(X_train , Y_train)
Y_test_pred = classifier.predict(X_test)
print("accuracy_score : " , accuracy_score(Y_test , Y_test_pred))
print("percision_score : " , precision_score(Y_test , Y_test_pred , pos_label='g'))
print("recall_score : " , recall_score(Y_test , Y_test_pred , pos_label='g'))
print("f1_score : " , f1_score(Y_test , Y_test_pred , pos_label='g'))
print("confusion_matrix : \n" , confusion_matrix(Y_test , Y_test_pred))



accuracy_score :  0.8075772681954138
percision_score :  0.7590788308237378
recall_score :  0.88259526261586
f1_score :  0.8161904761904762
confusion_matrix : 
 [[857 114]
 [272 763]]


In [34]:
# when using k = 3
# the lowest accuracy was (0.811) 
# the lowest f1_score was (0.821)
# so using k = 3 is not good for our model as it is underfitting the data.

# when using k = 5
# the model achieved high accuracy (0.846)
# the f1_score was (0.847) which is good as well.
# so using k = 5 is good for our model as it is not underfitting the data.
 
# when using k = 7
#This model achieved the absolute highest F1-Score (82.73%) and a fantastic Recall (88.9%)
# so using k = 7 is the best for our model as it is not underfitting the data and it has the highest f1_score.

#using k = 9 and k = 11
# the accuracy and f1_score are lower than k = 7
